# Week 6 on Colab — only if llama.cpp will not install on your machine

Monday's merge and conversion, done on Colab. It ends by giving you one file, `notes-q8_0.gguf`, to download. Serving that file is still done on your own machine, with the `llama-server` binary from a llama.cpp release.

No GPU is needed. This notebook makes the `q8_0` file only: the conversion script can write that level by itself. The `q4_k_m` comparison needs the `llama-quantize` program, which you run locally if you can.

## 1. Get your code

In [ ]:
import os

REPO_URL = "..."   # the address of YOUR repo

if not os.path.exists("course") and os.path.basename(os.getcwd()) != "course":
    !git clone {REPO_URL} course
if os.path.basename(os.getcwd()) != "course":
    %cd course
!git pull -q
!pip install -q -e . peft gguf sentencepiece protobuf
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp

## 2. Your adapter

It is on your Drive, inside last week's run folder. Change `RUN` to the run you scored.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
RUN = Path("/content/drive/MyDrive/tunelab/week-05/run-01")
assert (RUN / "adapter" / "adapter_config.json").exists(), f"no adapter in {RUN}"
OUT = Path("/content/drive/MyDrive/tunelab/week-06")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Merge, with your own `merge.py`

Into a **full-precision** copy of the base model, not the 4-bit one you trained on.

In [ ]:
import importlib.util

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

spec = importlib.util.spec_from_file_location("merge", "week-06/day-1/merge.py")
merge = importlib.util.module_from_spec(spec)
spec.loader.exec_module(merge)

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
base = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)

merged = merge.merge_adapter(base, str(RUN / "adapter"))
merge.save_merged(merged, tokenizer, "/content/merged")
print(type(merged).__name__, f"{sum(p.numel() for p in merged.parameters()):,} parameters")

## 4. Convert

Straight to `q8_0`: about 530 MB.

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile "{OUT}/notes-q8_0.gguf" --outtype q8_0
!ls -la "{OUT}"

## 5. Take it home

It is on your Drive. This also offers it as a download.

In [ ]:
from google.colab import files

files.download(str(OUT / "notes-q8_0.gguf"))